# pyclesperanto NumPy API Compatibility

This notebook demonstrates how `pyclesperanto.Array` can use the NumPy array API.

Data lives on the GPU, but the same NumPy syntax works transparently.

This compatibility as limits, some related to GPU limitation and other to implementation choices:
* dtype: pyclesperanto does not support `bool`, replaced by `uint8`, and does not support 64b type which are silently cast to 32b equivalent type.
* nD array: pyclesperanto is limited to 1d, 2d, or 3d array, higher dimension are not possible, same for 0d array concept.
* dtype overflow: pyclesperanto perfome type saturation, e.g. `uint8(250) + uint8(10) = uint8(255)` instead of `uint8(250) + uint8(10) = uint8(5)`.
* Broadcasting: only scalar and singleton is supported, broadcasting two arrays to a common shape is not.

In [1]:
import numpy as np
import pyclesperanto as cle

cle.select_device()

(OpenCL) Apple M5 Max (OpenCL 1.2)
	Vendor:                      Apple
	Driver Version:              1.2 1.0
	Device Type:                 GPU
	Compute Units:               40
	Global Memory Size:          53084 MB
	Local Memory Size:           0 MB
	Maximum Buffer Size:         9953 MB
	Max Clock Frequency:         1000 MHz
	Image Support:               Yes

## 1. Array creation and conversion

Create device arrays from NumPy arrays (or the reverse) with `from_array` / `np.asarray`.

In [2]:
data = np.arange(12, dtype=np.float32).reshape(3, 4)
arr = cle.Array.from_array(data)
arr

array([[ 0.,  1.,  2.,  3.],
       [ 4.,  5.,  6.,  7.],
       [ 8.,  9., 10., 11.]], dtype=float32, mtype=buffer)

In [3]:
np.asarray(arr)

array([[ 0.,  1.,  2.,  3.],
       [ 4.,  5.,  6.,  7.],
       [ 8.,  9., 10., 11.]], dtype=float32)

In [4]:
cle.zeros((2, 3), dtype=np.float32), cle.ones((2, 3)), cle.full((2, 2), 7)

(array([[0., 0., 0.],
        [0., 0., 0.]], dtype=float32, mtype=buffer),
 array([[1., 1., 1.],
        [1., 1., 1.]], dtype=float32, mtype=buffer),
 array([[7, 7],
        [7, 7]], dtype=int32, mtype=buffer))

In [5]:
cle.arange(0, 10, 2), cle.linspace(0, 1, num=5), cle.eye(3)

(array([0, 2, 4, 6, 8], dtype=int32, mtype=buffer),
 array([0.  , 0.25, 0.5 , 0.75, 1.  ], dtype=float32, mtype=buffer),
 array([[1., 0., 0.],
        [0., 1., 0.],
        [0., 0., 1.]], dtype=float32, mtype=buffer))

## 2. Attributes

Familiar NumPy attributes: `shape`, `dtype`, `size`, `nbytes`, `T`.

In [6]:
arr.shape, arr.dtype, arr.size, arr.nbytes

((3, 4), dtype('float32'), 12, 48)

In [7]:
arr.T

array([[ 0.,  4.,  8.],
       [ 1.,  5.,  9.],
       [ 2.,  6., 10.],
       [ 3.,  7., 11.]], dtype=float32, mtype=buffer)

## 3. Element-wise operators

Arithmetic, comparison and bitwise operators work directly on device arrays, with scalars, other
device arrays, or broadcasting.

In [8]:
a = cle.Array.from_array(np.asarray([1.0, 2.0, 3.0, 4.0], dtype=np.float32))
b = cle.Array.from_array(np.asarray([4.0, 3.0, 2.0, 1.0], dtype=np.float32))

a + b, a - b, a * b, a / b

(array([5., 5., 5., 5.], dtype=float32, mtype=buffer),
 array([-3., -1.,  1.,  3.], dtype=float32, mtype=buffer),
 array([4., 6., 6., 4.], dtype=float32, mtype=buffer),
 array([0.25     , 0.6666667, 1.5      , 4.       ], dtype=float32, mtype=buffer))

In [9]:
a ** 2, a % 3, a // 2

(array([ 1.,  4.,  9., 16.], dtype=float32, mtype=buffer),
 array([1., 2., 0., 1.], dtype=float32, mtype=buffer),
 array([0., 1., 1., 2.], dtype=float32, mtype=buffer))

In [10]:
a > b, a == b

(array([0, 0, 1, 1], dtype=uint8, mtype=buffer),
 array([0, 0, 0, 0], dtype=uint8, mtype=buffer))

In [11]:
row = cle.Array.from_array(np.asarray([[10.0, 20.0, 30.0, 40.0]], dtype=np.float32))
arr + row

array([[10., 21., 32., 43.],
       [14., 25., 36., 47.],
       [18., 29., 40., 51.]], dtype=float32, mtype=buffer)

## 4. NumPy ufuncs

NumPy universal functions (`np.sqrt`, `np.add`, ...) dispatch to device kernels via `__array_ufunc__`.

In [12]:
np.sqrt(a), np.exp(a), np.sin(a)

(array([1.       , 1.4142135, 1.7320508, 2.       ], dtype=float32, mtype=buffer),
 array([ 2.718282,  7.389056, 20.085537, 54.598152], dtype=float32, mtype=buffer),
 array([ 0.841471  ,  0.9092974 ,  0.14112002, -0.7568025 ], dtype=float32, mtype=buffer))

In [13]:
np.add(a, b), np.maximum(a, b), np.power(a, 2)

(array([5., 5., 5., 5.], dtype=float32, mtype=buffer),
 array([4., 3., 3., 4.], dtype=float32, mtype=buffer),
 array([ 1.,  4.,  9., 16.], dtype=float32, mtype=buffer))

In [14]:
out = cle.Array.empty(a.shape, dtype=np.float32)
np.multiply(a, 2.0, out=out)
out

array([2., 4., 6., 8.], dtype=float32, mtype=buffer)

## 5. Comparisons

Comparisons return device arrays (`uint8` in place of `bool`), comparable directly with NumPy arrays.

In [15]:
a > 2.0, np.equal(a, b)

(array([0, 0, 1, 1], dtype=uint8, mtype=buffer),
 array([0, 0, 0, 0], dtype=uint8, mtype=buffer))

## 6. Reductions

Standard reductions (`sum`, `mean`, `min`, `max`, `std`, `argmax`, ...) support `axis` and `keepdims`,
either as methods or as NumPy free functions.

In [16]:
data3d = np.arange(24, dtype=np.float32).reshape(2, 3, 4)
arr3d = cle.Array.from_array(data3d)

arr3d.sum(), arr3d.mean(), arr3d.max(), arr3d.argmax()

(np.float32(276.0), np.float32(11.5), np.float32(23.0), np.uint32(23))

In [17]:
arr3d.sum(axis=1), arr3d.sum(axis=1, keepdims=True).shape

(array([[12., 15., 18., 21.],
        [48., 51., 54., 57.]], dtype=float32, mtype=buffer),
 (2, 1, 4))

In [18]:
np.sum(arr3d, axis=(0, 1))

array([60., 66., 72., 78.], dtype=float32, mtype=buffer)

## 7. Cumulative operations

`np.ufunc.accumulate` (e.g. `np.add.accumulate`, cumulative sum) works along a given axis.

In [19]:
data2d = np.asarray([[1.0, 2.0], [3.0, 4.0]], dtype=np.float32)
arr2d = cle.Array.from_array(data2d)

np.add.accumulate(arr2d, axis=0), np.multiply.accumulate(arr2d, axis=1)

(array([[1., 2.],
        [4., 6.]], dtype=float32, mtype=buffer),
 array([[ 1.,  2.],
        [ 3., 12.]], dtype=float32, mtype=buffer))

## 8. Slicing and indexing

Standard NumPy-style slicing, including assignment, is supported.

In [20]:
arr3d[0], arr3d[:, 1, :], arr3d[0, 0, 0]

(array([[ 0.,  1.,  2.,  3.],
        [ 4.,  5.,  6.,  7.],
        [ 8.,  9., 10., 11.]], dtype=float32, mtype=buffer),
 array([[ 4.,  5.,  6.,  7.],
        [16., 17., 18., 19.]], dtype=float32, mtype=buffer),
 0.0)

In [21]:
sliced = cle.Array.from_array(data2d.copy())
sliced[0, :] = 9.0
sliced

array([[9., 9.],
       [3., 4.]], dtype=float32, mtype=buffer)

## 9. Reshaping and other methods

`reshape`, `ravel`, `flatten`, `squeeze`, `copy` mirror their NumPy counterparts.

In [22]:
arr3d.reshape(4, 6).shape, arr3d.ravel().shape, arr3d.flatten().shape

((4, 6), (24,), (24,))

In [23]:
cle.Array.empty((1, 3)).squeeze().shape

(3,)

## 10. dtype casting

`astype` follows NumPy semantics, including `copy=False` no-op for matching dtypes.

In [24]:
a.astype(np.int32), a.astype(np.float32, copy=False) is a

(array([1, 2, 3, 4], dtype=int32, mtype=buffer), True)